In [16]:

import pandas as pd
from pathlib import Path

# Project directory (notebooks/landslide_ml.ipynb is inside notebooks/)
BASE_DIR = Path.cwd()

# If the notebook is running from the notebooks folder, move to project root
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Load your existing CSV
filename = BASE_DIR / "data" / "raw" / "Global_Landslide_Catalog_Export_rows.csv"

df = pd.read_csv(filename, low_memory=False)

# Keep useful columns
keep_cols = [
    "event_id",
    "event_date",
    "landslide_trigger",
    "landslide_category",
    "landslide_size",
    "longitude",
    "latitude",
    "country_name",
    "admin_division_name"
]

df = df[[c for c in keep_cols if c in df.columns]].copy()

# Convert coordinates and dates
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

# Remove records without coordinates
df = df.dropna(subset=["latitude", "longitude"])

# Northeast Indian states
northeast_states = [
    "arunachal pradesh", "assam", "manipur", "meghalaya",
    "mizoram", "nagaland", "sikkim", "tripura"
]

df["admin_clean"] = (
    df["admin_division_name"].astype(str).str.strip().str.lower()
)
df["country_clean"] = (
    df["country_name"].astype(str).str.strip().str.lower()
)

# Filter Northeast India
ne_df = df[
    (df["country_clean"] == "india") &
    (df["admin_clean"].isin(northeast_states))
].copy()

# Display results
print("Original records:", len(df))
print("Northeast India records:", len(ne_df))

display(ne_df.head())

print("\nAvailable landslide triggers:")
display(ne_df["landslide_trigger"].value_counts(dropna=False))

# Save the filtered dataset
output_file = BASE_DIR / "data" / "raw" / "northeast_landslide_events.csv"
ne_df.drop(columns=["admin_clean", "country_clean"]).to_csv(
    output_file, index=False
)

print("\nSaved file:", output_file)


C:\Users\ARYAN-PC\AppData\Local\Temp\ipykernel_27320\985725287.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")


Original records: 11033
Northeast India records: 251


,event_id,event_date,landslide_trigger,landslide_category,landslide_size,longitude,latitude,country_name,admin_division_name,admin_clean,country_clean
59,2164,2010-07-29 23:00:00,downpour,landslide,medium,94.482147,25.309863,India,Manipur,manipur,india
524,7370,2015-08-01 00:00:00,continuous_rain,landslide,medium,94.107800,24.141900,India,Manipur,manipur,india
653,1109,2009-08-20 00:00:00,downpour,landslide,medium,93.164500,25.521500,India,Assam,assam,india
713,322,2007-10-21 00:00:00,rain,landslide,medium,93.433300,24.750000,India,Assam,assam,india
760,5552,2013-09-24 00:00:00,unknown,landslide,medium,94.361900,25.098500,India,Manipur,manipur,india



Available landslide triggers:


landslide_trigger
downpour           104
continuous_rain     64
rain                60
unknown             11
monsoon              9
mining               3
Name: count, dtype: int64


Saved file: c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\data\raw\northeast_landslide_events.csv


In [17]:

# Check the quality of the Northeast India dataset

print("Total records:", len(ne_df))
print("Unique event IDs:", ne_df["event_id"].nunique())

print("\nMissing values:")
print(ne_df.isna().sum())

print("\nEvent dates:")
print("Earliest:", ne_df["event_date"].min())
print("Latest:", ne_df["event_date"].max())

print("\nRecords by state:")
print(ne_df["admin_division_name"].value_counts())

print("\nLandslide categories:")
print(ne_df["landslide_category"].value_counts(dropna=False))

print("\nLandslide sizes:")
print(ne_df["landslide_size"].value_counts(dropna=False))

print("\nCoordinate ranges:")
print("Latitude:", ne_df["latitude"].min(), "to", ne_df["latitude"].max())
print("Longitude:", ne_df["longitude"].min(), "to", ne_df["longitude"].max())


Total records: 251
Unique event IDs: 251

Missing values:
event_id               0
event_date             0
landslide_trigger      0
landslide_category     0
landslide_size         0
longitude              0
latitude               0
country_name           0
admin_division_name    0
admin_clean            0
country_clean          0
dtype: int64

Event dates:
Earliest: 2007-05-23 00:00:00
Latest: 2016-10-15 00:00:00

Records by state:
admin_division_name
Assam                82
Manipur              56
Sikkim               31
Mizoram              27
Arunachal Pradesh    20
Meghalaya            18
Nagaland             14
Tripura               3
Name: count, dtype: int64

Landslide categories:
landslide_category
landslide              221
mudslide                24
complex                  3
rock_fall                2
translational_slide      1
Name: count, dtype: int64

Landslide sizes:
landslide_size
medium        202
small          29
large          15
unknown         3
very_large      2

In [4]:

import pandas as pd

# Start from your filtered Northeast India dataset
model_df = ne_df.copy()

# Keep useful historical event information
model_df = model_df[
    [
        "event_id",
        "event_date",
        "latitude",
        "longitude",
        "admin_division_name",
        "landslide_trigger",
        "landslide_size",
    ]
].copy()

# Ensure correct data types
model_df["event_date"] = pd.to_datetime(
    model_df["event_date"], errors="coerce"
)
model_df["latitude"] = pd.to_numeric(
    model_df["latitude"], errors="coerce"
)
model_df["longitude"] = pd.to_numeric(
    model_df["longitude"], errors="coerce"
)

# Extract seasonal features
model_df["year"] = model_df["event_date"].dt.year
model_df["month"] = model_df["event_date"].dt.month

# Remove invalid rows and duplicate event IDs
model_df = model_df.dropna(
    subset=["event_date", "latitude", "longitude", "month"]
)
model_df = model_df.drop_duplicates(subset=["event_id"])

print("Prepared event records:", len(model_df))
print(model_df.head())

model_df.to_csv(
    "northeast_landslide_events_clean.csv", index=False
)


Prepared event records: 251
     event_id          event_date   latitude  longitude admin_division_name  \
59       2164 2010-07-29 23:00:00  25.309863  94.482147             Manipur   
524      7370 2015-08-01 00:00:00  24.141900  94.107800             Manipur   
653      1109 2009-08-20 00:00:00  25.521500  93.164500               Assam   
713       322 2007-10-21 00:00:00  24.750000  93.433300               Assam   
760      5552 2013-09-24 00:00:00  25.098500  94.361900             Manipur   

    landslide_trigger landslide_size  year  month  
59           downpour         medium  2010      7  
524   continuous_rain         medium  2015      8  
653          downpour         medium  2009      8  
713              rain         medium  2007     10  
760           unknown         medium  2013      9  


In [18]:

import requests
import pandas as pd
import time

weather_rows = []

for i, row in model_df.iterrows():
    event_date = row["event_date"].strftime("%Y-%m-%d")
    start_date = (
        row["event_date"] - pd.Timedelta(days=7)
    ).strftime("%Y-%m-%d")

    params = {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": start_date,
        "end_date": event_date,
        "daily": (
            "precipitation_sum,"
            "temperature_2m_mean,"
            "relative_humidity_2m_mean"
        ),
        "timezone": "Asia/Kolkata",
        "models": "era5_land"
    }

    try:
        response = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
        daily = data.get("daily", {})

        dates = daily.get("time", [])
        rainfall = daily.get("precipitation_sum", [])
        temperature = daily.get("temperature_2m_mean", [])
        humidity = daily.get("relative_humidity_2m_mean", [])

        if dates:
            weather_rows.append({
                "event_id": row["event_id"],
                "event_date": event_date,
                "latitude": row["latitude"],
                "longitude": row["longitude"],
                "rainfall_event_day_mm": rainfall[-1],
                "rainfall_3day_mm": sum(
                    x or 0 for x in rainfall[-3:]
                ),
                "rainfall_7day_mm": sum(
                    x or 0 for x in rainfall
                ),
                "temperature_c": temperature[-1],
                "humidity_percent": humidity[-1]
            })

    except Exception as e:
        print("Failed event", row["event_id"], ":", str(e))

    if len(weather_rows) % 25 == 0 and weather_rows:
        print("Fetched:", len(weather_rows), "/", len(model_df))

weather_df = pd.DataFrame(weather_rows)
weather_df.to_csv("landslide_events_with_weather.csv", index=False)

print("\nWeather records fetched:", len(weather_df))
print(weather_df.head())
print("\nMissing values:\n", weather_df.isna().sum())


Fetched: 25 / 251
Fetched: 50 / 251
Fetched: 75 / 251
Fetched: 100 / 251
Fetched: 125 / 251
Fetched: 150 / 251
Fetched: 175 / 251
Failed event 6970 : ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Fetched: 200 / 251
Fetched: 225 / 251
Fetched: 250 / 251

Weather records fetched: 250
   event_id  event_date   latitude  longitude rainfall_event_day_mm  \
0      2164  2010-07-29  25.309863  94.482147                  None   
1      7370  2015-08-01  24.141900  94.107800                  None   
2      1109  2009-08-20  25.521500  93.164500                  None   
3       322  2007-10-21  24.750000  93.433300                  None   
4      5552  2013-09-24  25.098500  94.361900                  None   

   rainfall_3day_mm  rainfall_7day_mm  temperature_c  humidity_percent  
0                 0                 0           20.3                91  
1                 0                 0           19.4 

In [6]:

import requests
import pandas as pd

row = model_df.iloc[0]
event_date = row["event_date"].strftime("%Y-%m-%d")
start_date = (
    row["event_date"] - pd.Timedelta(days=7)
).strftime("%Y-%m-%d")

params = {
    "latitude": float(row["latitude"]),
    "longitude": float(row["longitude"]),
    "start_date": start_date,
    "end_date": event_date,
    "daily": "precipitation_sum",
    "timezone": "Asia/Kolkata",
    "models": "era5_land"
}

response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params=params,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Request URL:", response.url)
print("API response:")
print(response.text[:3000])


HTTP status: 200
Request URL: https://archive-api.open-meteo.com/v1/archive?latitude=25.3098633&longitude=94.48214677&start_date=2010-07-22&end_date=2010-07-29&daily=precipitation_sum&timezone=Asia%2FKolkata&models=era5_land
API response:
{"latitude":25.300003,"longitude":94.5,"generationtime_ms":0.02765655517578125,"utc_offset_seconds":19800,"timezone":"Asia/Kolkata","timezone_abbreviation":"GMT+5:30","elevation":1559.0,"daily_units":{"time":"iso8601","precipitation_sum":"mm"},"daily":{"time":["2010-07-22","2010-07-23","2010-07-24","2010-07-25","2010-07-26","2010-07-27","2010-07-28","2010-07-29"],"precipitation_sum":[null,null,null,null,null,null,null,null]}}


In [8]:

import requests
import pandas as pd
import time
from pathlib import Path

session = requests.Session()
weather_rows = []
failed = []

for i, (_, row) in enumerate(model_df.iterrows(), start=1):
    event_date = row["event_date"]
    start_date = (event_date - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    end_date = event_date.strftime("%Y-%m-%d")

    params = {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": start_date,
        "end_date": end_date,
        "daily": (
            "precipitation_sum,"
            "temperature_2m_mean,"
            "relative_humidity_2m_mean"
        ),
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    try:
        response = session.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        daily = response.json().get("daily", {})

        rain = daily.get("precipitation_sum", [])
        temp = daily.get("temperature_2m_mean", [])
        humidity = daily.get("relative_humidity_2m_mean", [])

        def safe_sum(values):
            # Do not treat missing rainfall as zero.
            if len(values) != 7 or any(v is None for v in values):
                return None
            return sum(values)

        rain_3day = rain[-3:] if len(rain) >= 3 else []
        rain_7day = rain[-7:] if len(rain) >= 7 else []

        weather_rows.append({
            "event_id": row["event_id"],
            "event_date": end_date,
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "rainfall_event_day_mm": (
                rain[-1] if rain and rain[-1] is not None else None
            ),
            "rainfall_3day_mm": (
                sum(rain_3day)
                if len(rain_3day) == 3
                and all(v is not None for v in rain_3day)
                else None
            ),
            "rainfall_7day_mm": safe_sum(rain_7day),
            "temperature_c": (
                temp[-1] if temp and temp[-1] is not None else None
            ),
            "humidity_percent": (
                humidity[-1]
                if humidity and humidity[-1] is not None else None
            )
        })

    except Exception as e:
        failed.append({"event_id": row["event_id"], "error": str(e)})

    if i % 25 == 0:
        print(f"Processed {i}/{len(model_df)} events")

    time.sleep(0.1)

weather_df = pd.DataFrame(weather_rows)

# Save the CSV inside the notebooks folder
OUTPUT_FILE = Path.cwd() / "landslide_events_with_weather.csv"
weather_df.to_csv(OUTPUT_FILE, index=False)

print("\nSuccessful API records:", len(weather_df))
print("Failed requests:", len(failed))
print("\nSaved file:", OUTPUT_FILE)
print("\nMissing values:\n", weather_df.isna().sum())
print("\nRainfall sample:")
print(weather_df[
    ["event_id", "rainfall_event_day_mm",
     "rainfall_3day_mm", "rainfall_7day_mm"]
].head(10))


Processed 25/251 events
Processed 50/251 events
Processed 75/251 events
Processed 100/251 events
Processed 125/251 events
Processed 150/251 events
Processed 175/251 events
Processed 200/251 events
Processed 225/251 events
Processed 250/251 events

Successful API records: 251
Failed requests: 0

Saved file: c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\notebooks\landslide_events_with_weather.csv

Missing values:
 event_id                 0
event_date               0
latitude                 0
longitude                0
rainfall_event_day_mm    0
rainfall_3day_mm         0
rainfall_7day_mm         0
temperature_c            0
humidity_percent         0
dtype: int64

Rainfall sample:
   event_id  rainfall_event_day_mm  rainfall_3day_mm  rainfall_7day_mm
0      2164                   27.7              42.9             125.3
1      7370                   17.3             304.9             443.5
2      1109                    2.7               9.9              90.1
3       322 

In [9]:

import pandas as pd
import numpy as np

events = model_df.copy()
events["event_date"] = pd.to_datetime(events["event_date"])

rng = np.random.default_rng(42)
event_dates_by_location = {}

for _, row in events.iterrows():
    key = (round(row["latitude"], 3), round(row["longitude"], 3))
    event_dates_by_location.setdefault(key, []).append(row["event_date"])

negative_rows = []

for _, row in events.iterrows():
    key = (round(row["latitude"], 3), round(row["longitude"], 3))
    known_dates = event_dates_by_location[key]

    # Try up to 50 dates from the same year.
    candidates = pd.date_range(
        f"{row['year']}-01-01",
        f"{row['year']}-12-31",
        freq="D"
    )

    chosen = None
    for _ in range(50):
        candidate = candidates[rng.integers(0, len(candidates))]
        if all(abs((candidate - d).days) > 14 for d in known_dates):
            chosen = candidate
            break

    if chosen is not None:
        negative_rows.append({
            "event_id": row["event_id"],
            "event_date": chosen,
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "admin_division_name": row["admin_division_name"],
            "landslide_risk": 0
        })

positive_rows = events[[
    "event_id", "event_date", "latitude", "longitude",
    "admin_division_name"
]].copy()
positive_rows["landslide_risk"] = 1

negative_df = pd.DataFrame(negative_rows)
positive_df = positive_rows.copy()

print("Positive event examples:", len(positive_df))
print("Candidate non-event examples:", len(negative_df))
print(negative_df.head())


Positive event examples: 251
Candidate non-event examples: 251
   event_id event_date   latitude  longitude admin_division_name  \
0      2164 2010-02-02  25.309863  94.482147             Manipur   
1      7370 2015-10-10  24.141900  94.107800             Manipur   
2      1109 2009-06-10  25.521500  93.164500               Assam   
3       322 2007-06-08  24.750000  93.433300               Assam   
4      5552 2013-11-10  25.098500  94.361900             Manipur   

   landslide_risk  
0               0  
1               0  
2               0  
3               0  
4               0  


In [10]:

import requests
import pandas as pd
import time

session = requests.Session()
negative_weather_rows = []
failed_negative = []

for i, (_, row) in enumerate(negative_df.iterrows(), start=1):
    end_date = pd.to_datetime(row["event_date"])
    start_date = (end_date - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    end_date_str = end_date.strftime("%Y-%m-%d")

    params = {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": start_date,
        "end_date": end_date_str,
        "daily": (
            "precipitation_sum,"
            "temperature_2m_mean,"
            "relative_humidity_2m_mean"
        ),
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    try:
        response = session.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params,
            timeout=30
        )
        response.raise_for_status()
        daily = response.json().get("daily", {})

        rain = daily.get("precipitation_sum", [])
        temp = daily.get("temperature_2m_mean", [])
        humidity = daily.get("relative_humidity_2m_mean", [])

        def total_if_complete(values):
            if len(values) == 7 and all(v is not None for v in values):
                return sum(values)
            return None

        rain3 = rain[-3:] if len(rain) >= 3 else []
        negative_weather_rows.append({
            "event_id": row["event_id"],
            "event_date": end_date_str,
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "rainfall_event_day_mm": (
                rain[-1] if rain and rain[-1] is not None else None
            ),
            "rainfall_3day_mm": (
                sum(rain3) if len(rain3) == 3
                and all(v is not None for v in rain3) else None
            ),
            "rainfall_7day_mm": total_if_complete(rain),
            "temperature_c": (
                temp[-1] if temp and temp[-1] is not None else None
            ),
            "humidity_percent": (
                humidity[-1] if humidity and humidity[-1] is not None else None
            ),
            "landslide_risk": 0
        })

    except Exception as e:
        failed_negative.append({
            "event_id": row["event_id"], "error": str(e)
        })

    if i % 25 == 0:
        print(f"Processed {i}/{len(negative_df)} comparison dates")

    time.sleep(0.1)

negative_weather_df = pd.DataFrame(negative_weather_rows)

print("\nWeather records:", len(negative_weather_df))
print("Failed requests:", len(failed_negative))
print("\nMissing values:\n", negative_weather_df.isna().sum())
print(negative_weather_df.head())


Processed 25/251 comparison dates
Processed 50/251 comparison dates
Processed 75/251 comparison dates
Processed 100/251 comparison dates
Processed 125/251 comparison dates
Processed 150/251 comparison dates
Processed 175/251 comparison dates
Processed 200/251 comparison dates
Processed 225/251 comparison dates
Processed 250/251 comparison dates

Weather records: 251
Failed requests: 0

Missing values:
 event_id                 0
event_date               0
latitude                 0
longitude                0
rainfall_event_day_mm    0
rainfall_3day_mm         0
rainfall_7day_mm         0
temperature_c            0
humidity_percent         0
landslide_risk           0
dtype: int64
   event_id  event_date   latitude  longitude  rainfall_event_day_mm  \
0      2164  2010-02-02  25.309863  94.482147                    0.0   
1      7370  2015-10-10  24.141900  94.107800                    9.1   
2      1109  2009-06-10  25.521500  93.164500                    0.0   
3       322  2007-06-08

In [11]:

# Add labels to actual recorded landslide events
positive_weather_df = weather_df.merge(
    model_df[["event_id", "admin_division_name"]],
    on="event_id",
    how="left"
)

positive_weather_df["landslide_risk"] = 1

# Add month from the date
positive_weather_df["month"] = pd.to_datetime(
    positive_weather_df["event_date"]
).dt.month

negative_weather_df["month"] = pd.to_datetime(
    negative_weather_df["event_date"]
).dt.month

# Combine recorded events and candidate comparison dates
combined_df = pd.concat(
    [positive_weather_df, negative_weather_df],
    ignore_index=True
)

# Keep only useful columns for machine learning
feature_columns = [
    "latitude",
    "longitude",
    "rainfall_event_day_mm",
    "rainfall_3day_mm",
    "rainfall_7day_mm",
    "temperature_c",
    "humidity_percent",
    "month"
]

X = combined_df[feature_columns]
y = combined_df["landslide_risk"]

print("Combined dataset shape:", combined_df.shape)
print("\nClass distribution:")
print(y.value_counts())
print("\nMissing values:", X.isna().sum().sum())
print("\nFinal dataset preview:")
display(combined_df.head())


Combined dataset shape: (502, 12)

Class distribution:
landslide_risk
1    251
0    251
Name: count, dtype: int64

Missing values: 0

Final dataset preview:


,event_id,event_date,latitude,longitude,rainfall_event_day_mm,rainfall_3day_mm,rainfall_7day_mm,temperature_c,humidity_percent,admin_division_name,landslide_risk,month
0,2164,2010-07-29,25.309863,94.482147,27.7,42.9,125.3,20.7,92,Manipur,1,7
1,7370,2015-08-01,24.141900,94.107800,17.3,304.9,443.5,20.6,94,Manipur,1,8
2,1109,2009-08-20,25.521500,93.164500,2.7,9.9,90.1,27.0,83,Assam,1,8
3,322,2007-10-21,24.750000,93.433300,1.4,17.4,66.3,21.9,83,Assam,1,10
4,5552,2013-09-24,25.098500,94.361900,5.7,7.3,15.6,18.9,87,Manipur,1,9


In [12]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Select input features and target
X = combined_df[feature_columns]
y = combined_df["landslide_risk"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, predictions), 3),
        "Precision": round(precision_score(y_test, predictions), 3),
        "Recall": round(recall_score(y_test, predictions), 3),
        "F1 Score": round(f1_score(y_test, predictions), 3),
        "ROC-AUC": round(roc_auc_score(y_test, probabilities), 3)
    })

    print(f"\n{'=' * 45}")
    print(name)
    print("=" * 45)
    print(classification_report(y_test, predictions))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

results_df = pd.DataFrame(results)
display(results_df)



Logistic Regression
              precision    recall  f1-score   support

           0       0.72      0.65      0.68        51
           1       0.67      0.74      0.70        50

    accuracy                           0.69       101
   macro avg       0.70      0.69      0.69       101
weighted avg       0.70      0.69      0.69       101

Confusion Matrix:
[[33 18]
 [13 37]]

Random Forest
              precision    recall  f1-score   support

           0       0.81      0.59      0.68        51
           1       0.67      0.86      0.75        50

    accuracy                           0.72       101
   macro avg       0.74      0.72      0.72       101
weighted avg       0.74      0.72      0.72       101

Confusion Matrix:
[[30 21]
 [ 7 43]]


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.693,0.673,0.74,0.705,0.776
1,Random Forest,0.723,0.672,0.86,0.754,0.809


In [14]:

import joblib
import os
import json
from pathlib import Path

# Set project root (notebooks/landslide_ml.ipynb)
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Create folders in project root
os.makedirs(BASE_DIR / "models", exist_ok=True)
os.makedirs(BASE_DIR / "data" / "processed", exist_ok=True)

# Train the selected model on the full experimental dataset
best_model = models["Random Forest"]
best_model.fit(X, y)

# Save model and feature names
joblib.dump(
    best_model,
    BASE_DIR / "models" / "random_forest_landslide.pkl"
)

joblib.dump(
    feature_columns,
    BASE_DIR / "models" / "landslide_features.pkl"
)

# Save the processed dataset
combined_df.to_csv(
    BASE_DIR / "data" / "processed" / "landslide_dataset.csv",
    index=False
)

# Save project metadata
metadata = {
    "model": "Random Forest",
    "features": list(feature_columns),
    "target": "landslide_risk",
    "records": int(len(combined_df)),
    "warning": (
        "Experimental dataset: comparison dates are not verified "
        "landslide-free observations. Not for official warnings."
    )
}

with open(BASE_DIR / "models" / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Model and project files saved successfully!")

print("\nFiles created:")
print(BASE_DIR / "models" / "random_forest_landslide.pkl")
print(BASE_DIR / "models" / "landslide_features.pkl")
print(BASE_DIR / "models" / "model_metadata.json")
print(BASE_DIR / "data" / "processed" / "landslide_dataset.csv")


Model and project files saved successfully!

Files created:
c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\models\random_forest_landslide.pkl
c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\models\landslide_features.pkl
c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\models\model_metadata.json
c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\data\processed\landslide_dataset.csv


In [15]:

import pandas as pd
from pathlib import Path

# Locate project folders
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    PROJECT_DIR = BASE_DIR.parent
else:
    PROJECT_DIR = BASE_DIR
    BASE_DIR = PROJECT_DIR / "notebooks"

# Read your existing CSV file
filename = (
    PROJECT_DIR / "data" / "raw" /
    "Global_Landslide_Catalog_Export_rows.csv"
)

df = pd.read_csv(filename, low_memory=False)

# Keep useful columns available in the historical event catalog
keep_cols = [
    "event_id",
    "event_date",
    "landslide_trigger",
    "landslide_category",
    "landslide_size",
    "longitude",
    "latitude",
    "country_name",
    "admin_division_name"
]

df = df[[c for c in keep_cols if c in df.columns]].copy()

# Check required columns
required_cols = [
    "latitude", "longitude",
    "country_name", "admin_division_name"
]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Convert coordinates and dates
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

# Remove records without coordinates
df = df.dropna(subset=["latitude", "longitude"])

# Keep Northeast India records by state/administrative division
northeast_states = [
    "arunachal pradesh", "assam", "manipur", "meghalaya",
    "mizoram", "nagaland", "sikkim", "tripura"
]

df["admin_clean"] = (
    df["admin_division_name"].astype(str).str.strip().str.lower()
)
df["country_clean"] = (
    df["country_name"].astype(str).str.strip().str.lower()
)

ne_df = df[
    (df["country_clean"] == "india") &
    (df["admin_clean"].isin(northeast_states))
].copy()

print("Original records:", len(df))
print("Northeast India records:", len(ne_df))
display(ne_df.head())

print("\nAvailable triggers:")
display(ne_df["landslide_trigger"].value_counts(dropna=False))

# Save the output inside the notebooks folder
output_file = BASE_DIR / "northeast_landslide_events.csv"
ne_df.drop(columns=["admin_clean", "country_clean"]).to_csv(
    output_file, index=False
)

print("\nFiltered dataset saved to:", output_file)


C:\Users\ARYAN-PC\AppData\Local\Temp\ipykernel_27320\2089463235.py:49: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")


Original records: 11033
Northeast India records: 251


,event_id,event_date,landslide_trigger,landslide_category,landslide_size,longitude,latitude,country_name,admin_division_name,admin_clean,country_clean
59,2164,2010-07-29 23:00:00,downpour,landslide,medium,94.482147,25.309863,India,Manipur,manipur,india
524,7370,2015-08-01 00:00:00,continuous_rain,landslide,medium,94.107800,24.141900,India,Manipur,manipur,india
653,1109,2009-08-20 00:00:00,downpour,landslide,medium,93.164500,25.521500,India,Assam,assam,india
713,322,2007-10-21 00:00:00,rain,landslide,medium,93.433300,24.750000,India,Assam,assam,india
760,5552,2013-09-24 00:00:00,unknown,landslide,medium,94.361900,25.098500,India,Manipur,manipur,india



Available triggers:


landslide_trigger
downpour           104
continuous_rain     64
rain                60
unknown             11
monsoon              9
mining               3
Name: count, dtype: int64


Filtered dataset saved to: c:\Users\ARYAN-PC\Desktop\rugvedi\landslide-risk-prediction\notebooks\northeast_landslide_events.csv
